# Simple CatBoost Baseline for Playground Series S6E9

This notebook is a clean, CPU-friendly starting point for predicting `Will_Buy_EV`. It uses only the competition data, CatBoost, and stratified 5-fold cross-validation.

**What you get:**

- explicit conversion of the competition's `Yes` / `No` target labels;
- native CatBoost categorical handling with no feature engineering;
- fold-by-fold and overall out-of-fold ROC AUC;
- feature importance and a reusable OOF file; and
- a validated `submission.csv` ready to download.

It does **not** use target encoding, external data, blending, pseudo-labels, or leaderboard-driven tuning. Standard 5-fold cross-validation provides honest out-of-fold predictions, with training capped at 600 trees per fold.

> **Version note:** the competition target is stored as `Yes` / `No`. This revision converts it explicitly before training and adds reusable OOF predictions.

## 1. Imports and paths

On Kaggle, add the competition as an input. For a local run, place the three CSV files in `../data/raw/`.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from IPython.display import Markdown, display
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = 'Will_Buy_EV'
ID_COLUMN = 'id'

KAGGLE_DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e9')
LOCAL_DATA_DIR = Path('../data/raw')
REQUIRED_FILES = ('train.csv', 'test.csv', 'sample_submission.csv')

DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.is_dir() else LOCAL_DATA_DIR

if not all((DATA_DIR / filename).is_file() for filename in REQUIRED_FILES):
    raise FileNotFoundError(
        'Could not find train.csv, test.csv, and sample_submission.csv. '
        'For Kaggle, attach the Playground Series S6E9 competition data as an input.'
    )

print(f'Using data from: {DATA_DIR}')

## 2. Load the competition data

The target arrives as `Yes` / `No`, so we map it once to `1` / `0`. The identifier is retained for alignment but excluded from model features.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

assert TARGET in train.columns
assert ID_COLUMN in train.columns and ID_COLUMN in test.columns
assert sample_submission.columns.tolist() == [ID_COLUMN, TARGET]

target_map = {'No': 0, 'Yes': 1}
observed_labels = set(train[TARGET].dropna().unique())
if train[TARGET].isna().any() or observed_labels != set(target_map):
    raise ValueError(f'Expected Yes/No target labels, found: {sorted(observed_labels)}')
train[TARGET] = train[TARGET].map(target_map).astype('int8')

feature_columns = [column for column in test.columns if column != ID_COLUMN]
train_feature_columns = [
    column for column in train.columns if column not in {ID_COLUMN, TARGET}
]
assert train_feature_columns == feature_columns

dataset_summary = pd.DataFrame(
    {
        'value': [len(train), len(test), len(train.columns) - 2, train[TARGET].mean()],
    },
    index=['train rows', 'test rows', 'model features', 'positive rate'],
)
display(dataset_summary)

fig, ax = plt.subplots(figsize=(5, 3))
train[TARGET].value_counts().sort_index().plot.bar(ax=ax, color=['#4C78A8', '#F58518'])
ax.set(title='Target distribution', xlabel=TARGET, ylabel='Rows')
ax.set_xticklabels(['No', 'Yes'], rotation=0)
plt.show()

display(train.head())
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Positive rate: {train[TARGET].mean():.4f}')

## 3. Prepare features

CatBoost handles numeric missing values natively. We only replace missing categorical values with an explicit label; no fitted preprocessing or feature engineering is used.

In [ ]:
X = train[feature_columns].copy()
X_test = test[feature_columns].copy()
y = train[TARGET].astype(int).to_numpy()

categorical_columns = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)

feature_summary = pd.DataFrame(
    {
        'feature type': ['numeric', 'categorical'],
        'count': [len(numeric_columns), len(categorical_columns)],
    }
)
display(feature_summary)
print(f'Categorical features: {categorical_columns}')

## 4. Stratified 5-fold CatBoost

Every training row is predicted by a model that did not fit on that row, while test predictions are averaged across folds. For simplicity, each validation fold is also used for early stopping; use the same protocol when comparing changes.

In [ ]:
training_started = perf_counter()
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_predictions = np.zeros(len(train), dtype=float)
test_predictions = np.zeros(len(test), dtype=float)
fold_assignments = np.full(len(train), -1, dtype=np.int8)
fold_scores = []
fold_sizes = []
best_iterations = []
feature_importances = np.zeros(len(feature_columns), dtype=float)
test_pool = Pool(X_test, cat_features=categorical_columns)

for fold, (train_index, valid_index) in enumerate(cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y[train_index], y[valid_index]

    train_pool = Pool(X_train, y_train, cat_features=categorical_columns)
    valid_pool = Pool(X_valid, y_valid, cat_features=categorical_columns)

    model = CatBoostClassifier(
        iterations=600,
        learning_rate=0.05,
        depth=8,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=SEED + fold,
        thread_count=-1,
        allow_writing_files=False,
        verbose=100,
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=75,
        use_best_model=True,
    )

    valid_prediction = model.predict_proba(valid_pool)[:, 1]
    oof_predictions[valid_index] = valid_prediction
    fold_assignments[valid_index] = fold
    test_predictions += model.predict_proba(test_pool)[:, 1] / N_SPLITS
    feature_importances += model.get_feature_importance(train_pool) / N_SPLITS

    fold_auc = roc_auc_score(y_valid, valid_prediction)
    best_iteration = model.get_best_iteration() + 1
    fold_scores.append(fold_auc)
    fold_sizes.append(len(valid_index))
    best_iterations.append(best_iteration)
    print(f'Fold {fold}: AUC = {fold_auc:.6f} | best iteration = {best_iteration}')

assert (fold_assignments > 0).all()
assert np.isfinite(oof_predictions).all()
assert np.isfinite(test_predictions).all()
training_minutes = (perf_counter() - training_started) / 60

## 5. Evaluate OOF predictions

Use this OOF score as the local reference for extensions that keep the same folds and protocol. Avoid repeatedly tuning features or blend weights against the public leaderboard.

In [ ]:
overall_auc = roc_auc_score(y, oof_predictions)
fold_summary = pd.DataFrame(
    {
        'fold': range(1, N_SPLITS + 1),
        'validation rows': fold_sizes,
        'best iteration': best_iterations,
        'ROC AUC': fold_scores,
    }
)
display(fold_summary.round({'ROC AUC': 6}))
display(Markdown(
    f'### Overall OOF ROC AUC: `{overall_auc:.6f}`  '
    f'\nFold mean ± std: `{np.mean(fold_scores):.6f} ± {np.std(fold_scores, ddof=1):.6f}`  '
    f'\nMeasured training time: `{training_minutes:.1f} minutes`'
))

fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y, oof_predictions, name=f'CatBoost OOF (AUC={overall_auc:.4f})', ax=ax)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set_title('Out-of-fold ROC curve')
plt.show()

## 6. Inspect model signals

The chart below is the mean built-in CatBoost feature importance across folds. It is a diagnostic, not a causal interpretation.

In [ ]:
importance_table = (
    pd.DataFrame({'feature': feature_columns, 'importance': feature_importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
display(importance_table.head(15).round({'importance': 3}))

plot_table = importance_table.head(15).sort_values('importance')
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(plot_table['feature'], plot_table['importance'], color='#4C78A8')
ax.set(title='Top 15 mean CatBoost feature importances', xlabel='Importance')
plt.show()

## 7. Sensible next steps

Once this baseline is reproduced, useful extensions include:

1. testing one target-independent feature idea at a time;
2. tuning regularization or tree depth with the same CV folds;
3. comparing a second model family using full OOF predictions; and
4. blending only after checking OOF correlation and fixing the weight before a final evaluation.

Keep the test set out of feature selection, and treat the leaderboard as a final check rather than a tuning set.

## 8. Save submission and OOF predictions

The notebook writes `submission.csv` and full OOF predictions to `/kaggle/working/`. Uploading or submitting remains a separate manual step.

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = test_predictions

assert submission[ID_COLUMN].equals(test[ID_COLUMN])
assert submission[TARGET].between(0.0, 1.0).all()
assert np.isfinite(submission[TARGET]).all()

output_dir = (
    Path('/kaggle/working')
    if Path('/kaggle/working').exists()
    else Path('../submissions')
)
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission.csv'
oof_path = output_dir / 'catboost_oof.csv'

oof_frame = pd.DataFrame(
    {
        ID_COLUMN: train[ID_COLUMN],
        'target': y,
        'fold': fold_assignments,
        'prediction': oof_predictions,
    }
)
submission.to_csv(submission_path, index=False)
oof_frame.to_csv(oof_path, index=False)

display(submission.head())
print('Saved files:')
for path in (submission_path, oof_path):
    print(f'  - {path.resolve()}')

## Final notes

This is intentionally a transparent baseline: raw features, one model family, one validation protocol, and no leaderboard-based tuning. It should be easy to fork and improve without hidden steps.

If this notebook saved you time or gave you a useful starting point, an upvote is appreciated. Good luck with S6E9!